# 06 · A série histórica: 2006 a 2026 (hipóteses C1, C2, C4, C5 e a exploratória C3)

**Testes confirmatórios.** Especificações registradas em `docs/hipoteses.md` (seção "Série
histórica") antes de qualquer análise da série.

Até aqui tudo comparava 2022 com 2026. Com os seis 1ºs turnos de 2006 a 2026, dá para ver
o que é novo e o que é padrão:

| pleito | candidato do PT | adversário principal |
|---|---|---|
| 2006 | Lula | Geraldo Alckmin (PSDB) |
| 2010 | Dilma Rousseff | José Serra (PSDB) |
| 2014 | Dilma Rousseff | Aécio Neves (PSDB) |
| 2018 | Fernando Haddad | Jair Bolsonaro (PSL) |
| 2022 | Lula | Jair Bolsonaro (PL) |
| 2026 | Lula | Flávio Bolsonaro (PL) |

Unidade: AMC (5.562, as mesmas fronteiras em todos os pleitos). Medida: fatia do PT entre os
dois maiores (`pt_2p`), em logit. **Limite:** as covariáveis são do Censo 2022 (religião: o
Censo mais próximo, 2010 até 2014 e 2022 a partir de 2018); para os pleitos antigos elas
medem a estrutura dos municípios, não o retrato exato da época.

In [ ]:
import warnings

import altair as alt
import numpy as np
import polars as pl
import pyfixest as pf
from sklearn.cluster import KMeans

from kingsrace import paths, resultados
from kingsrace.features.base import geografia_amc
from kingsrace.viz import tema  # noqa: F401  (registra e ativa o tema do projeto)
from kingsrace.viz.tema import DIVERGENTE, SERIES, TINTA

resultados.configura_altair()
live = resultados.abre("06_serie")  # métricas e gráficos em resultados/06_serie/
explora = resultados.params("exploracao")  # escolhas da C3, livres para `dvc exp run -S`
warnings.filterwarnings("ignore", message=".*singleton fixed effect")

P = paths.PROCESSED
ANOS = [2006, 2010, 2014, 2018, 2022, 2026]
amc = pl.read_parquet(P / "painel_amc.parquet").filter(pl.col("turno") == 1)
geo = geografia_amc(pl.read_parquet(P / "municipios.parquet"))
cov = pl.read_parquet(P / "covariaveis_amc.parquet").with_columns(
    log_renda=pl.col("renda_pc_mediana").log()
)
serie = amc.join(geo, on="amc_id").join(cov, on="amc_id", how="left")
cluster = {"CRV1": "cd_regiao_imediata"}
zero = alt.Chart().mark_rule(color=TINTA["apagada"], strokeDash=[4, 4]).encode(y=alt.datum(0))

## Contexto: o PT no país, pleito a pleito

Fatia do PT entre os dois maiores no 1º turno, no Brasil (sem exterior).

In [ ]:
nacional = (
    amc.group_by("ano")
    .agg(pl.col("votos_pt", "votos_adv").sum())
    .with_columns(pt_2p=100 * pl.col("votos_pt") / (pl.col("votos_pt") + pl.col("votos_adv")))
    .sort("ano")
)
for linha in nacional.iter_rows(named=True):
    resultados.metrica(live, f"nacional/{linha['ano']}/pt_2p_pct", linha["pt_2p"])
resultados.grafico(
    live,
    "nacional",
    (
        alt.Chart(nacional)
        .mark_line(
            point=alt.OverlayMarkDef(size=70, filled=True, color=DIVERGENTE["pt"]),
            color=DIVERGENTE["pt"],
        )
        .encode(
            x=alt.X("ano:O", title=None, axis=alt.Axis(labelAngle=0)),
            y=alt.Y(
                "pt_2p:Q",
                title="PT entre os dois maiores, 1º turno (%)",
                scale=alt.Scale(domain=[35, 65]),
            ),
            tooltip=[
                alt.Tooltip("ano:O", title="Ano"),
                alt.Tooltip("pt_2p:Q", title="PT (%)", format=".1f"),
            ],
        )
        + alt.Chart().mark_rule(color=TINTA["apagada"], strokeDash=[4, 4]).encode(y=alt.datum(50))
    ).properties(width=520, height=220, title="O PT entre os dois maiores, 1º turno"),
)

## C1 · A clivagem do lulismo ao longo do tempo

Para cada pleito, uma regressão do voto no PT (logit) em características dos municípios,
**padronizadas** (o coeficiente é o efeito de 1 desvio-padrão; o de Nordeste é a diferença
para as demais regiões). Sem efeitos fixos: aqui interessa a geografia nacional.

**Hipóteses registradas:** (a) renda negativa em todos os pleitos; (b) |renda| maior em
2022 que em 2006; (c) % evangélica mais negativa em 2022 e 2026 que em 2006.

In [ ]:
VARS = {
    "log_renda": "Renda per capita (log)",
    "evangelica": "% evangélica",
    "taxa_alfabetizacao": "Alfabetização",
    "pct_urbana": "% urbana",
    "idade_mediana": "Idade mediana",
}
linhas = []
for ano in ANOS:
    religiao = "pct_evangelica_2010" if ano <= 2014 else "pct_evangelica"
    d = serie.filter(pl.col("ano") == ano).with_columns(
        evangelica=pl.col(religiao), nordeste=(pl.col("regiao") == "NE").cast(pl.Int8)
    )
    d = d.drop_nulls(["pt_2p_logit", *VARS]).with_columns(
        [((pl.col(v) - pl.col(v).mean()) / pl.col(v).std()).alias(f"z_{v}") for v in VARS]
    )
    fit = pf.feols(
        "pt_2p_logit ~ " + " + ".join(f"z_{v}" for v in VARS) + " + nordeste",
        d.to_pandas(),
        vcov=cluster,
    )
    t = fit.tidy()
    resultados.metrica(live, f"C1/{ano}/r2", fit._r2)
    for termo, rotulo in [*((f"z_{v}", r) for v, r in VARS.items()), ("nordeste", "Nordeste")]:
        chave = termo.removeprefix("z_")
        resultados.metrica(live, f"C1/{ano}/{chave}", t.loc[termo, "Estimate"])
        linhas.append(
            {
                "ano": ano,
                "chave": chave,
                "variavel": rotulo,
                "coef": t.loc[termo, "Estimate"],
                "ic_inf": t.loc[termo, "2.5%"],
                "ic_sup": t.loc[termo, "97.5%"],
                "r2": fit._r2,
            }
        )
c1 = pl.DataFrame(linhas)
for (chave,), tabela in c1.group_by("chave", maintain_order=True):
    resultados.coeficientes(live, f"C1_{chave}", tabela, rotulo="ano")


def c1_coef(chave, ano):
    return c1.filter((pl.col("chave") == chave) & (pl.col("ano") == ano))["coef"].item()


resultados.metricas(
    live,
    "C1",
    {
        "a_sustenta": c1.filter(pl.col("chave") == "log_renda")["ic_sup"].max() < 0,
        "b_sustenta": abs(c1_coef("log_renda", 2022)) > abs(c1_coef("log_renda", 2006)),
        "c_sustenta": max(c1_coef("evangelica", 2022), c1_coef("evangelica", 2026))
        < c1_coef("evangelica", 2006),
    },
)
c1.pivot(on="variavel", index="ano", values="coef").with_columns(pl.all().exclude("ano").round(3))

**Como ler:** um painel por característica; cada ponto é o coeficiente num pleito, com IC
de 95%. Abaixo de zero, a característica está associada a **menos** voto no PT.

In [ ]:
ordem_vars = [
    "Renda per capita (log)",
    "% evangélica",
    "Nordeste",
    "Alfabetização",
    "% urbana",
    "Idade mediana",
]
base_c1 = alt.Chart().encode(x=alt.X("ano:O", title=None, axis=alt.Axis(labelAngle=0)))
resultados.grafico(
    live,
    "C1_clivagem",
    alt.layer(
        zero,
        base_c1.mark_rule(strokeWidth=2, color=SERIES[0]).encode(
            y=alt.Y("ic_inf:Q", title="Coeficiente (logit)"), y2="ic_sup:Q"
        ),
        base_c1.mark_line(color=SERIES[0], strokeWidth=1.5).encode(y="coef:Q"),
        base_c1.mark_circle(size=50, opacity=1, color=SERIES[0]).encode(
            y="coef:Q",
            tooltip=[
                alt.Tooltip("variavel:N", title="Variável"),
                alt.Tooltip("ano:O", title="Ano"),
                alt.Tooltip("coef:Q", title="Coeficiente", format="+.3f"),
                alt.Tooltip("ic_inf:Q", title="IC inf.", format="+.3f"),
                alt.Tooltip("ic_sup:Q", title="IC sup.", format="+.3f"),
            ],
        ),
        data=c1,
    )
    .properties(width=200, height=150)
    .facet(facet=alt.Facet("variavel:N", sort=ordem_vars, title=None), columns=3)
    .resolve_scale(y="independent")
    .properties(title="A clivagem do voto no PT, 2006–2026"),
)

**Resultado:**

- **(a) sustenta-se:** a renda é negativa em todos os pleitos. Municípios mais pobres votam
  mais no PT desde 2006, com qualquer candidato.
- **(b) sustenta-se, por pouco:** |renda| 0,57 em 2022 contra 0,49 em 2006 (os ICs se
  sobrepõem; a clivagem de renda é forte e estável, mais do que crescente).
- **(c) sustenta-se:** o efeito evangélico **não existia** em 2006–2014 e aparece em 2018
  (−0,15), mantendo-se em 2022 e 2026. É o efeito Bolsonaro.
- **Exploratório:** o quanto essas características explicam o voto (R²) subiu de ~0,55 em
  2006–2010 para ~0,80 a partir de 2018: o voto ficou mais "previsível" pela estrutura social.
- **Exploratório, e ligado à C5 abaixo:** com o resto fixo, a **% urbana** era negativa para o
  PT de 2010 a 2018 e fica **positiva em 2022 e 2026**; a idade mediana também passa a pesar a
  favor do PT. As áreas urbanas e mais velhas se reaproximaram do PT depois de 2018.

In [ ]:
c1.group_by("ano").agg(pl.col("r2").first().round(2)).sort("ano")

## C2 · O voto ficou mais regional?

A variação do voto no PT entre AMCs, dividida em três níveis: **entre UFs**, **entre regiões
imediatas da mesma UF** e **dentro da região imediata** (ponderada pelo eleitorado).
**Hipótese registrada:** a fração entre UFs é maior em 2022 que em 2006.

In [ ]:
linhas = []
for ano in ANOS:
    d = serie.filter(pl.col("ano") == ano).with_columns(
        m_uf=(pl.col("pt_2p_logit") * pl.col("aptos")).sum().over("uf")
        / pl.col("aptos").sum().over("uf"),
        m_ri=(pl.col("pt_2p_logit") * pl.col("aptos")).sum().over("cd_regiao_imediata")
        / pl.col("aptos").sum().over("cd_regiao_imediata"),
    )
    w = d["aptos"].to_numpy().astype(float)
    y, m_uf, m_ri = (d[c].to_numpy() for c in ("pt_2p_logit", "m_uf", "m_ri"))
    media = np.average(y, weights=w)
    total = np.average((y - media) ** 2, weights=w)
    for nivel, chave, valor in [
        ("Entre UFs", "entre_uf", np.average((m_uf - media) ** 2, weights=w)),
        ("Entre regiões imediatas da UF", "entre_ri", np.average((m_ri - m_uf) ** 2, weights=w)),
        ("Dentro da região imediata", "dentro_ri", np.average((y - m_ri) ** 2, weights=w)),
    ]:
        resultados.metrica(live, f"C2/{ano}/{chave}", valor / total)
        linhas.append(
            {
                "ano": ano,
                "nivel": nivel,
                "variancia": valor,
                "fracao": valor / total,
                "total": total,
            }
        )
    resultados.metrica(live, f"C2/{ano}/variancia_total", total)
c2 = pl.DataFrame(linhas)


def c2_entre_uf(ano):
    return c2.filter((pl.col("ano") == ano) & (pl.col("nivel") == "Entre UFs"))["fracao"].item()


resultados.metrica(live, "C2/sustenta", c2_entre_uf(2022) > c2_entre_uf(2006))
c2.pivot(on="nivel", index="ano", values="fracao").with_columns(pl.all().exclude("ano").round(3))

**Como ler:** a altura total de cada barra é a variância do voto no PT entre AMCs (quanto os
lugares diferem entre si); as cores mostram em que nível estão as diferenças.

In [ ]:
niveis = ["Entre UFs", "Entre regiões imediatas da UF", "Dentro da região imediata"]
resultados.grafico(
    live,
    "C2_variancia",
    (
        alt.Chart(c2)
        .mark_bar(size=34)
        .encode(
            x=alt.X("ano:O", title=None, axis=alt.Axis(labelAngle=0)),
            y=alt.Y("variancia:Q", title="Variância do logit do PT entre AMCs"),
            color=alt.Color(
                "nivel:N", sort=niveis, scale=alt.Scale(domain=niveis, range=SERIES[:3]), title=None
            ),
            order=alt.Order("ordem:Q"),
            tooltip=[
                alt.Tooltip("ano:O", title="Ano"),
                alt.Tooltip("nivel:N", title="Nível"),
                alt.Tooltip("fracao:Q", title="Fração do total", format=".1%"),
                alt.Tooltip("total:Q", title="Variância total", format=".3f"),
            ],
        )
        .transform_calculate(ordem=f"indexof({niveis}, datum.nivel)")
        .properties(
            width=480, height=260, title="Mais diferença dentro dos estados; o pico de 2018"
        )
    ),
)

**Resultado: C2 não se sustenta como formulada.** A fração **entre UFs caiu** (70% em 2006,
62% em 2022) e a fração **entre regiões imediatas da mesma UF subiu** (15% → 25%). O voto
ficou mais regional, mas **dentro dos estados**: o interior se separou das capitais e das
regiões metropolitanas. A variância total atinge o pico em 2018 (Haddad, muito concentrado
no Nordeste) e é a menor da série em 2026.

## C4 · As variações entre pleitos foram uniformes?

Para cada par de 1ºs turnos consecutivos: o desvio-padrão da variação do PT entre AMCs
(quanto os lugares se moveram de forma diferente) e a correlação entre os dois pleitos.
**Hipótese registrada:** 2022→2026 é mais uniforme que a média das transições anteriores.

In [ ]:
largo = serie.pivot(
    on="ano", index=["amc_id", "cd_regiao_imediata"], values=["pt_2p_logit", "aptos"]
)
linhas_c4, linhas_c5 = [], []
for a0, a1 in zip(ANOS[:-1], ANOS[1:], strict=True):
    d = largo.with_columns(
        dl=pl.col(f"pt_2p_logit_{a1}") - pl.col(f"pt_2p_logit_{a0}"),
        log_aptos=pl.col(f"aptos_{a0}").log(),
        nivel=pl.col(f"pt_2p_logit_{a0}"),
    )
    par = f"{a0}→{a1}"
    linhas_c4.append(
        {
            "transicao": par,
            "dp_variacao": d["dl"].std(),
            "correlacao": float(np.corrcoef(d[f"pt_2p_logit_{a0}"], d[f"pt_2p_logit_{a1}"])[0, 1]),
            "variacao_media": d["dl"].mean(),
        }
    )
    t = pf.feols("dl ~ log_aptos + nivel | cd_regiao_imediata", d.to_pandas(), vcov=cluster).tidy()
    linhas_c5.append(
        {
            "transicao": par,
            "coef": t.loc["log_aptos", "Estimate"],
            "ic_inf": t.loc["log_aptos", "2.5%"],
            "ic_sup": t.loc["log_aptos", "97.5%"],
        }
    )
c4 = pl.DataFrame(linhas_c4)
for linha in c4.iter_rows(named=True):
    resultados.metricas(
        live,
        f"C4/{resultados.chave(linha['transicao'])}",
        {k: linha[k] for k in ("dp_variacao", "correlacao", "variacao_media")},
    )
resultados.metrica(live, "C4/sustenta", c4["dp_variacao"][-1] < c4["dp_variacao"][:-1].mean())
c4.with_columns(pl.all().exclude("transicao").round(3))

**Resultado: C4 sustenta-se, com folga.** O desvio-padrão de 2022→2026 (0,14) é cerca de um
terço do das transições anteriores (0,40–0,49), e a correlação entre os dois pleitos (0,987) é
a maior da série. Em 2026 o país se moveu quase em bloco. Uma razão provável: é a única
transição com **o mesmo candidato do PT** dos dois lados.

## C5 · O gradiente de tamanho é novo?

A pergunta que começou o projeto: municípios menores se afastaram mais do PT entre 2022 e
2026. Isso acontece sempre? Para cada transição, o efeito do tamanho (log do eleitorado)
sobre a variação do PT, comparando AMCs da mesma região imediata e com o mesmo nível de
partida. **Hipótese registrada:** positivo em pelo menos 3 das 4 transições anteriores.

**Como ler:** acima de zero, os municípios **maiores** se moveram mais a favor do PT (ou
perderam menos); abaixo de zero, os **menores**.

In [ ]:
c5 = pl.DataFrame(linhas_c5).with_columns(
    sinal=pl.when(pl.col("coef") > 0)
    .then(pl.lit("maiores a favor do PT"))
    .otherwise(pl.lit("menores a favor do PT"))
)
for linha in c5.iter_rows(named=True):
    resultados.metricas(
        live,
        f"C5/{resultados.chave(linha['transicao'])}",
        {k: linha[k] for k in ("coef", "ic_inf", "ic_sup")},
    )
positivas = int((c5["ic_inf"][:4] > 0).sum())  # as 4 transições antes de 2022→2026
resultados.metricas(live, "C5", {"positivas_antes_2022": positivas, "sustenta": positivas >= 3})
resultados.coeficientes(live, "C5_gradiente", c5, rotulo="transicao")
base_c5 = alt.Chart(c5).encode(
    x=alt.X("transicao:N", title=None, sort=None, axis=alt.Axis(labelAngle=0)),
    tooltip=[
        alt.Tooltip("transicao:N", title="Transição"),
        alt.Tooltip("coef:Q", title="Efeito do tamanho", format="+.4f"),
        alt.Tooltip("ic_inf:Q", title="IC inf.", format="+.4f"),
        alt.Tooltip("ic_sup:Q", title="IC sup.", format="+.4f"),
    ],
)
resultados.grafico(
    live,
    "C5_gradiente",
    (
        zero
        + base_c5.mark_rule(strokeWidth=2, color=SERIES[0]).encode(
            y=alt.Y("ic_inf:Q", title="Efeito de log(eleitorado) sobre Δ logit do PT"),
            y2="ic_sup:Q",
        )
        + base_c5.mark_circle(size=90, opacity=1, color=SERIES[0]).encode(y="coef:Q")
    ).properties(width=520, height=260, title="O gradiente de tamanho inverteu em 2018"),
)

In [ ]:
c5.select("transicao", pl.col("coef", "ic_inf", "ic_sup").round(4))

**Resultado: C5 não se sustenta, e é o achado mais interessante da série.** O gradiente de
2022→2026 **não é um padrão permanente**: de 2006 a 2018 o sinal era o oposto. As cidades
grandes se afastavam mais do PT (ou os municípios pequenos se aproximavam mais),
com o pico em 2014→2018, o desmoronamento urbano do PT. Em 2018→2022 o sinal **inverte**: as
cidades grandes voltaram ao PT com mais força. Em 2022→2026, continua: os pequenos se
afastaram mais. A pergunta deixa de ser "por que os pequenos?" e passa a ser **"o que mudou
por volta de 2018?"**.

## C3 · Tipos de trajetória (exploratório)

Agrupamento das AMCs pela **forma** da trajetória do PT: em cada pleito, a distância (em
logit) para a média nacional daquele ano, sem o nível médio de cada AMC (só importa como ela
se moveu). k-means com 4 grupos. Exploratório: não há hipótese testada aqui.

In [ ]:
media_nacional = amc.group_by("ano").agg(
    ((pl.col("pt_2p_logit") * pl.col("aptos")).sum() / pl.col("aptos").sum()).alias("nacional")
)
relativo = (
    amc.join(media_nacional, on="ano")
    .with_columns(rel=pl.col("pt_2p_logit") - pl.col("nacional"))
    .pivot(on="ano", index="amc_id", values="rel")
    .sort("amc_id")
)
X = relativo.select([str(a) for a in ANOS]).to_numpy()
grupos = (
    KMeans(explora["c3_grupos"], n_init=20, random_state=explora["c3_semente"])
    .fit(X - X.mean(axis=1, keepdims=True))
    .labels_
)
trajetorias = relativo.with_columns(grupo=pl.Series(grupos)).join(geo, on="amc_id")

perfil = (
    trajetorias.group_by("grupo")
    .agg(
        pl.len().alias("AMCs"),
        *[pl.col(str(a)).mean() for a in ANOS],
        (100 * (pl.col("regiao") == "NE").mean()).round(0).alias("% Nordeste"),
        (100 * (pl.col("regiao") == "S").mean()).round(0).alias("% Sul"),
    )
    .with_columns(subida=pl.col("2018") - pl.col("2006"), descida=pl.col("2026") - pl.col("2018"))
)


# nomes descritivos a partir da forma média
def nomeia(r):
    if r["2026"] < r["2006"] - 0.3:
        return "Afastamento gradual"
    if r["2018"] > 1.2:
        return "Viraram redutos do PT"
    if r["subida"] > 0.5:
        return "Subiram até 2018 e voltaram"
    return "Antipetistas que se aproximaram"


perfil = perfil.sort("2018", descending=True)
tipos = [nomeia(r) for r in perfil.iter_rows(named=True)]
tipos = [  # com outro número de grupos, dois grupos podem cair no mesmo nome
    t if tipos.count(t) == 1 else f"{t} ({tipos[:i].count(t) + 1})" for i, t in enumerate(tipos)
]
perfil = perfil.with_columns(tipo=pl.Series(tipos))
for linha in perfil.iter_rows(named=True):
    resultados.metricas(
        live,
        f"C3/{resultados.chave(linha['tipo'])}",
        {"amcs": linha["AMCs"], "pct_nordeste": linha["% Nordeste"], "pct_sul": linha["% Sul"]},
    )
perfil.select("tipo", "AMCs", *[pl.col(str(a)).round(2) for a in ANOS], "% Nordeste", "% Sul")

**Como ler:** a trajetória média de cada tipo, como distância da média nacional (0 = como o
país; acima, mais PT que o país). Um painel por tipo.

In [ ]:
longo = perfil.unpivot(
    index=["tipo", "AMCs"], on=[str(a) for a in ANOS], variable_name="ano", value_name="rel"
)
linha_tipo = (
    alt.Chart()
    .mark_line(
        point=alt.OverlayMarkDef(size=40, filled=True, color=DIVERGENTE["pt"]),
        color=DIVERGENTE["pt"],
    )
    .encode(
        x=alt.X("ano:O", title=None, axis=alt.Axis(labelAngle=0)),
        y=alt.Y("rel:Q", title="PT em relação à média nacional (logit)"),
        tooltip=[
            alt.Tooltip("tipo:N", title="Tipo"),
            alt.Tooltip("ano:O", title="Ano"),
            alt.Tooltip("rel:Q", title="Distância da média", format="+.2f"),
            alt.Tooltip("AMCs:Q", title="AMCs no tipo"),
        ],
    )
)
media_zero = alt.Chart().mark_rule(color=TINTA["apagada"], strokeDash=[4, 4]).encode(y=alt.datum(0))
resultados.grafico(
    live,
    "C3_trajetorias",
    alt.layer(media_zero, linha_tipo, data=longo)
    .properties(width=230, height=160)
    .facet(facet=alt.Facet("tipo:N", title=None, sort=perfil["tipo"].to_list()), columns=2)
    .properties(title="Quatro tipos de trajetória do PT, 2006–2026"),
)

**O que observar:** dois tipos explicam o mapa atual. Os **redutos do PT** (sobretudo no
Nordeste) se formaram entre 2006 e 2018 e recuaram um pouco depois. O **afastamento
gradual** (mais no Sul, municípios maiores) começou no meio do mapa e se afastou do PT em
todos os pleitos. Pista para a próxima etapa: quem são esses municípios, e o que eles têm em
comum além de região e tamanho?

## Veredito

| hipótese | resultado |
|---|---|
| C1 (a) renda negativa sempre | sustenta-se |
| C1 (b) clivagem de renda maior em 2022 | sustenta-se por pouco (estável, mais que crescente) |
| C1 (c) efeito evangélico surge | sustenta-se: aparece em 2018 |
| C2 mais variância entre UFs | **não**: a diferença cresceu *dentro* dos estados |
| C4 2022→2026 mais uniforme | sustenta-se: a transição mais uniforme da série |
| C5 gradiente de tamanho permanente | **não**: inverteu de sinal em 2018 |

In [ ]:
live.end()